In [ ]:
# CELL 1 — Mount Drive + Setup Kaggle + Download Datasets

# 1. Mount Google Drive
from google.colab import drive, files
drive.mount('/content/drive')

# 2. Install Kaggle
!pip install kaggle -q

# 3. Upload your kaggle.json
print(" A file chooser will appear — select your kaggle.json file")
uploaded = files.upload()

# 4. Set up Kaggle credentials
import os
os.makedirs('/root/.config/kaggle', exist_ok=True)
!cp kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json
print(" Kaggle API ready!")

# 5. Create project folders in Drive
PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
print(" Project folders created in Drive!")

# 6. Download all 3 datasets
print("⬇ Downloading datasets...")

!kaggle datasets download -d hzaurali/wheat-data-of-last-12-years-in-punjab-pakistan -p {PROJECT_DIR}/data/raw --unzip
print(" Dataset 1 done — Wheat Punjab")

!kaggle datasets download -d patelris/crop-yield-prediction-dataset -p {PROJECT_DIR}/data/raw --unzip
print(" Dataset 2 done — Crop Yield")

!kaggle datasets download -d tahirnadeem/crop-district-level-dataset-punjab-pakistan -p {PROJECT_DIR}/data/raw --unzip
print(" Dataset 3 done — District Level Punjab")

print("\n All done! Check your Drive → CropYieldProject → data → raw")

In [ ]:
# CELL 2 — Fix kaggle.json + re-download failed datasets

import os
from google.colab import files

# Fix the filename issue (capital K → lowercase k)
!find /content -name "*.json" | grep -i kaggle

# Copy whichever version exists
!cp /content/Kaggle.json /root/.config/kaggle/kaggle.json 2>/dev/null || \
 cp /content/kaggle.json /root/.config/kaggle/kaggle.json 2>/dev/null

!chmod 600 /root/.config/kaggle/kaggle.json
!cat /root/.config/kaggle/kaggle.json
print(" kaggle.json fixed!")

In [ ]:
# CELL 3 — Download the 2 replacement datasets

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

# Dataset 1 - Wheat Punjab (correct working link)
!kaggle datasets download -d mhaiderali37/wheat-data-of-last-12-years-in-punjab-pakistan -p {PROJECT_DIR}/data/raw --unzip
print(" Dataset 1 done — Wheat Punjab")

# Dataset 3 - District Level Punjab (correct working link)
!kaggle datasets download -d talhanazir168/crop-dataset -p {PROJECT_DIR}/data/raw --unzip
print("Dataset 3 done — District Level Punjab")

# Show all files we have
import os
print("\n All files in raw folder:")
for f in os.listdir(f'{PROJECT_DIR}/data/raw'):
    size = os.path.getsize(f'{PROJECT_DIR}/data/raw/{f}')
    print(f"  {f}  —  {size/1024:.1f} KB")

In [ ]:
# CELL 4 — Explore all datasets

import pandas as pd
import os

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'
raw = f'{PROJECT_DIR}/data/raw'

# --- Dataset 1: Wheat Punjab ---
print("="*50)
print("DATASET 1 — Wheat Data Punjab")
print("="*50)
wheat = pd.read_excel(f'{raw}/Wheat Data.xlsx')
print(wheat.shape)
print(wheat.columns.tolist())
print(wheat.head(3))

# --- Dataset 2: Crop Yield (patelris) ---
print("\n" + "="*50)
print("DATASET 2 — Crop Yield Prediction (yield_df)")
print("="*50)
yield_df = pd.read_csv(f'{raw}/yield_df.csv')
print(yield_df.shape)
print(yield_df.columns.tolist())
print(yield_df.head(3))

# --- Dataset 3: District Level Punjab ---
print("\n" + "="*50)
print("DATASET 3 — District Level Punjab")
print("="*50)
district = pd.read_csv(f'{raw}/Crop(Distric level).csv')
print(district.shape)
print(district.columns.tolist())
print(district.head(3))

In [ ]:
# CELL 5 — Clean, filter and merge datasets

import pandas as pd
import numpy as np

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'
raw = f'{PROJECT_DIR}/data/raw'
processed = f'{PROJECT_DIR}/data/processed'

# ── DATASET 1: Wheat Punjab (for trend reference only) ──
wheat_raw = pd.read_excel(f'{raw}/Wheat Data.xlsx', header=1)
wheat_raw.columns = ['drop', 'market_year', 'area_1000ha', 'production_1000tons', 'yield_ton_ha']
wheat = wheat_raw.drop(columns=['drop']).dropna(subset=['market_year'])
wheat = wheat[wheat['market_year'] != 'Market Year'].reset_index(drop=True)
print(" Dataset 1 (Wheat trends):")
print(wheat)

# ── DATASET 2: Filter Pakistan crops ──
yield_df = pd.read_csv(f'{raw}/yield_df.csv')
pakistan = yield_df[yield_df['Area'] == 'Pakistan'].copy()
pakistan.columns = [c.strip() for c in pakistan.columns]
pakistan = pakistan.rename(columns={
    'Item': 'crop',
    'Year': 'year',
    'hg/ha_yield': 'yield_hg_ha',
    'average_rain_fall_mm_per_year': 'rainfall_mm',
    'pesticides_tonnes': 'pesticides_tonnes',
    'avg_temp': 'avg_temp'
})
pakistan = pakistan.drop(columns=['Unnamed: 0', 'Area'])
pakistan['crop'] = pakistan['crop'].str.lower().str.strip()
print(f"\n Dataset 2 (Pakistan only): {pakistan.shape}")
print(pakistan['crop'].unique())
print(pakistan.head(3))

# ── DATASET 3: District level Punjab ──
district = pd.read_csv(f'{raw}/Crop(Distric level).csv')
district = district.rename(columns={'label': 'crop'})
district['crop'] = district['crop'].str.lower().str.strip()
print(f"\n Dataset 3 (District Punjab): {district.shape}")
print(district['crop'].unique())
print(district.head(3))

# ── MERGE Dataset 2 + Dataset 3 on crop ──
# Get average soil/climate per crop from district data
district_agg = district.groupby('crop').agg({
    'N': 'mean', 'P': 'mean', 'K': 'mean',
    'temperature': 'mean', 'humidity': 'mean',
    'ph': 'mean', 'rainfall': 'mean'
}).reset_index()

merged = pakistan.merge(district_agg, on='crop', how='inner')
print(f"\nMerged dataset shape: {merged.shape}")
print(merged.columns.tolist())
print(merged.head(3))

# ── Check missing values ──
print("\n🔍 Missing values:")
print(merged.isnull().sum())

# ── Save cleaned files ──
wheat.to_csv(f'{processed}/wheat_trends.csv', index=False)
merged.to_csv(f'{processed}/merged_dataset.csv', index=False)
district.to_csv(f'{processed}/district_data.csv', index=False)

print("\n All cleaned files saved to Drive/processed folder!")
print(f"Final merged dataset: {merged.shape[0]} rows, {merged.shape[1]} columns")

In [ ]:
# CELL 6 — Feature Engineering + Final ML Dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'
processed = f'{PROJECT_DIR}/data/processed'
outputs = f'{PROJECT_DIR}/outputs'

df = pd.read_csv(f'{processed}/merged_dataset.csv')

# ── FEATURE ENGINEERING ──

# 1. Temp × Rainfall interaction (key agricultural factor)
df['temp_x_rainfall'] = df['avg_temp'] * df['rainfall_mm']

# 2. NPK total nutrient score
df['NPK_total'] = df['N'] + df['P'] + df['K']

# 3. Yield in Ton/Ha (convert from hg/ha — easier to interpret)
df['yield_ton_ha'] = df['yield_hg_ha'] / 10000

# 4. Encode crop as category
df['crop_encoded'] = pd.Categorical(df['crop']).codes
crop_mapping = dict(enumerate(pd.Categorical(df['crop']).categories))
print(" Crop encoding:", crop_mapping)

# 5. Drop redundant columns
df_ml = df.drop(columns=['yield_hg_ha', 'rainfall'])  # keep rainfall_mm, drop duplicate

print(f"\n Final ML dataset shape: {df_ml.shape}")
print(df_ml.columns.tolist())
print(df_ml.describe().round(2))

# ── SAVE ──
df_ml.to_csv(f'{processed}/final_ml_dataset.csv', index=False)
print("\n final_ml_dataset.csv saved!")

# ── CORRELATION HEATMAP (Python version for report) ──
plt.figure(figsize=(12, 8))
numeric_cols = df_ml.select_dtypes(include=np.number).columns
corr = df_ml[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlGn',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap — Crop Yield Features (Punjab, Pakistan)',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{outputs}/correlation_heatmap.png', dpi=150)
plt.show()
print(" Heatmap saved!")

# ── YIELD BY CROP BOXPLOT ──
plt.figure(figsize=(12, 5))
order = df_ml.groupby('crop')['yield_ton_ha'].median().sort_values(ascending=False).index
sns.boxplot(data=df_ml, x='crop', y='yield_ton_ha', order=order, palette='Set2')
plt.title('Yield Distribution by Crop — Pakistan', fontsize=13, fontweight='bold')
plt.xlabel('Crop')
plt.ylabel('Yield (Ton/Ha)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{outputs}/yield_by_crop_boxplot.png', dpi=150)
plt.show()
print(" Boxplot saved!")

# ── WHEAT YIELD TREND ──
wheat = pd.read_csv(f'{processed}/wheat_trends.csv')
wheat = wheat[wheat['market_year'] != 'Average']
plt.figure(figsize=(10, 4))
plt.plot(wheat['market_year'], wheat['yield_ton_ha'],
         marker='o', color='#2e8b57', linewidth=2.5, markersize=8)
plt.fill_between(range(len(wheat)), wheat['yield_ton_ha'], alpha=0.15, color='#2e8b57')
plt.title('Wheat Yield Trend — Punjab, Pakistan (2014–2024)', fontsize=13, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Yield (Ton/Ha)')
plt.xticks(range(len(wheat)), wheat['market_year'], rotation=30)
plt.tight_layout()
plt.savefig(f'{outputs}/wheat_yield_trend.png', dpi=150)
plt.show()
print(" Wheat trend chart saved!")

print("\n🎉 Feature engineering complete! 3 charts saved to Drive/outputs/")

In [ ]:
# CELL 7 — Train 3 Models + Compare Results

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib
import warnings
warnings.filterwarnings('ignore')

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'
processed = f'{PROJECT_DIR}/data/processed'
outputs   = f'{PROJECT_DIR}/outputs'
models_dir= f'{PROJECT_DIR}/models'

# ── Load data ──
df = pd.read_csv(f'{processed}/final_ml_dataset.csv')

FEATURES = ['year', 'rainfall_mm', 'pesticides_tonnes', 'avg_temp',
            'N', 'P', 'K', 'temperature', 'humidity', 'ph',
            'temp_x_rainfall', 'NPK_total', 'crop_encoded']
TARGET = 'yield_ton_ha'

X = df[FEATURES]
y = df[TARGET]

# ── Train/Test split (80/20) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# ── Scale for SVR ──
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Helper: evaluate model ──
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    r2   = r2_score(y_te, pred)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5,
                           scoring='r2').mean()
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  R²   : {r2:.4f}")
    print(f"  CV R²: {cv:.4f}")
    return {'Model': name, 'MAE': round(mae,4),
            'RMSE': round(rmse,4), 'R²': round(r2,4),
            'CV R²': round(cv,4)}, pred

# ── MODEL 1: Random Forest ──
rf = RandomForestRegressor(n_estimators=200, max_depth=10,
                           random_state=42, n_jobs=-1)
rf_res, rf_pred = evaluate('Random Forest', rf,
                            X_train, X_test, y_train, y_test)
joblib.dump(rf, f'{models_dir}/random_forest.pkl')

# ── MODEL 2: XGBoost ──
xgb = XGBRegressor(n_estimators=200, learning_rate=0.05,
                   max_depth=6, random_state=42,
                   verbosity=0)
xgb_res, xgb_pred = evaluate('XGBoost', xgb,
                               X_train, X_test, y_train, y_test)
joblib.dump(xgb, f'{models_dir}/xgboost.pkl')

# ── MODEL 3: SVR ──
svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
svr_res, svr_pred = evaluate('SVR', svr,
                              X_train_sc, X_test_sc, y_train, y_test)
joblib.dump(svr, f'{models_dir}/svr.pkl')
joblib.dump(scaler, f'{models_dir}/scaler.pkl')

# ── RESULTS TABLE ──
results = pd.DataFrame([rf_res, xgb_res, svr_res])
print("\n\n📊 MODEL COMPARISON TABLE")
print("="*55)
print(results.to_string(index=False))

# ── COMPARISON BAR CHART ──
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = ['MAE', 'RMSE', 'R²']
colors  = ['#2e8b57', '#e07b39', '#5b7fa6']

for i, metric in enumerate(metrics):
    axes[i].bar(results['Model'], results[metric],
                color=colors, edgecolor='white', width=0.5)
    axes[i].set_title(metric, fontsize=13, fontweight='bold')
    axes[i].set_ylabel(metric)
    axes[i].set_ylim(0, results[metric].max() * 1.3)
    for j, v in enumerate(results[metric]):
        axes[i].text(j, v + 0.01, str(v),
                     ha='center', fontsize=10, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Model Comparison — Crop Yield Prediction (Punjab, Pakistan)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{outputs}/model_comparison.png', dpi=150)
plt.show()
print(" Model comparison chart saved!")

# ── ACTUAL vs PREDICTED ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
preds = [rf_pred, xgb_pred, svr_pred]
names = ['Random Forest', 'XGBoost', 'SVR']
cols  = ['#2e8b57', '#e07b39', '#5b7fa6']

for i, (pred, name, col) in enumerate(zip(preds, names, cols)):
    axes[i].scatter(y_test, pred, alpha=0.6, color=col, edgecolors='white', s=50)
    mn = min(y_test.min(), pred.min())
    mx = max(y_test.max(), pred.max())
    axes[i].plot([mn, mx], [mn, mx], 'k--', linewidth=1.5)
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel('Actual Yield')
    axes[i].set_ylabel('Predicted Yield')

plt.suptitle('Actual vs Predicted — All 3 Models',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{outputs}/actual_vs_predicted.png', dpi=150)
plt.show()
print(" Actual vs Predicted chart saved!")

# Save results
results.to_csv(f'{outputs}/model_results.csv', index=False)
print("\n All 3 models trained and saved!")

In [ ]:
# CELL 8 — Stacking Ensemble + SHAP Explainability

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'
processed  = f'{PROJECT_DIR}/data/processed'
outputs    = f'{PROJECT_DIR}/outputs'
models_dir = f'{PROJECT_DIR}/models'

# ── Load data ──
df = pd.read_csv(f'{processed}/final_ml_dataset.csv')
FEATURES = ['year','rainfall_mm','pesticides_tonnes','avg_temp',
            'N','P','K','temperature','humidity','ph',
            'temp_x_rainfall','NPK_total','crop_encoded']
TARGET = 'yield_ton_ha'
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# ── STACKING ENSEMBLE ──
# SVR needs scaling — wrap in Pipeline
svr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1))
])

base_learners = [
    ('rf',  RandomForestRegressor(n_estimators=200, max_depth=10,
                                   random_state=42, n_jobs=-1)),
    ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05,
                          max_depth=6, random_state=42, verbosity=0)),
    ('svr', svr_pipe)
]

stack = StackingRegressor(
    estimators=base_learners,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

stack.fit(X_train, y_train)
stack_pred = stack.predict(X_test)

mae  = mean_absolute_error(y_test, stack_pred)
rmse = np.sqrt(mean_squared_error(y_test, stack_pred))
r2   = r2_score(y_test, stack_pred)
cv   = cross_val_score(stack, X_train, y_train, cv=5, scoring='r2').mean()

print("="*50)
print("  STACKING ENSEMBLE RESULTS")
print("="*50)
print(f"  MAE  : {mae:.4f}")
print(f"  RMSE : {rmse:.4f}")
print(f"  R²   : {r2:.4f}")
print(f"  CV R²: {cv:.4f}")

joblib.dump(stack, f'{models_dir}/stacking_ensemble.pkl')
print(" Stacking ensemble saved!")

# ── FINAL COMPARISON TABLE (all 4) ──
all_results = pd.DataFrame([
    {'Model':'Random Forest', 'MAE':0.0029,'RMSE':0.0071,'R²':0.9999,'CV R²':0.9997},
    {'Model':'XGBoost',       'MAE':0.0009,'RMSE':0.0023,'R²':1.0000,'CV R²':1.0000},
    {'Model':'SVR',           'MAE':0.0806,'RMSE':0.0933,'R²':0.9865,'CV R²':0.9730},
    {'Model':'Stacking',      'MAE':round(mae,4),'RMSE':round(rmse,4),
                               'R²':round(r2,4),'CV R²':round(cv,4)},
])
print("\n FINAL 4-MODEL COMPARISON")
print(all_results.to_string(index=False))
all_results.to_csv(f'{outputs}/final_model_results.csv', index=False)

# ── SHAP on XGBoost (best model) ──
print("\n Computing SHAP values...")
xgb_model = joblib.load(f'{models_dir}/xgboost.pkl')
explainer  = shap.TreeExplainer(xgb_model)
shap_vals  = explainer.shap_values(X_test)

# SHAP Summary Bar Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_test, plot_type='bar',
                  feature_names=FEATURES, show=False)
plt.title('SHAP Feature Importance — XGBoost\n(Crop Yield Prediction, Punjab Pakistan)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{outputs}/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("SHAP bar chart saved!")

# SHAP Beeswarm Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_test, feature_names=FEATURES, show=False)
plt.title('SHAP Beeswarm — Feature Impact on Yield',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{outputs}/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print(" SHAP beeswarm chart saved!")

# RF Feature Importance
rf_model = joblib.load(f'{models_dir}/random_forest.pkl')
importances = pd.Series(rf_model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#2e8b57', edgecolor='white')
plt.title('Random Forest — Feature Importance\n(Crop Yield Prediction, Punjab Pakistan)',
          fontsize=12, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig(f'{outputs}/rf_feature_importance.png', dpi=150)
plt.show()
print(" RF feature importance chart saved!")

print("\n Stacking ensemble + SHAP complete! All charts saved to Drive/outputs/")

In [ ]:
# CELL 9 — Save Streamlit app to Drive

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

app_code = '''
import streamlit as st
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt

# Load models
import os
BASE = os.path.dirname(os.path.abspath(__file__))

rf  = joblib.load(f'{BASE}/models/random_forest.pkl')
xgb = joblib.load(f'{BASE}/models/xgboost.pkl')
svr = joblib.load(f'{BASE}/models/svr.pkl')
scaler = joblib.load(f'{BASE}/models/scaler.pkl')
stack  = joblib.load(f'{BASE}/models/stacking_ensemble.pkl')

FEATURES = ["year","rainfall_mm","pesticides_tonnes","avg_temp",
            "N","P","K","temperature","humidity","ph",
            "temp_x_rainfall","NPK_total","crop_encoded"]

st.set_page_config(page_title="Crop Yield Predictor",
                   page_icon="", layout="wide")

st.title(" Crop Yield Prediction — Punjab, Pakistan")
st.markdown("### Multi-Model Ensemble Approach for Climate-Resilient Agriculture")
st.markdown("---")

# Sidebar inputs
st.sidebar.header(" Enter Field Parameters")

crop = st.sidebar.selectbox("Crop Type", ["Wheat", "Maize"])
year = st.sidebar.slider("Year", 1990, 2025, 2020)
rainfall = st.sidebar.slider("Rainfall (mm/year)", 200, 1500, 494)
pesticides = st.sidebar.slider("Pesticides (tonnes)", 0, 30000, 6000)
avg_temp = st.sidebar.slider("Average Temperature (°C)", 15.0, 35.0, 24.0)
N = st.sidebar.slider("Nitrogen (N)", 0, 140, 56)
P = st.sidebar.slider("Phosphorus (P)", 0, 100, 42)
K = st.sidebar.slider("Potassium (K)", 0, 120, 61)
temperature = st.sidebar.slider("Soil Temperature (°C)", 15.0, 35.0, 24.0)
humidity = st.sidebar.slider("Humidity (%)", 50.0, 100.0, 86.0)
ph = st.sidebar.slider("Soil pH", 4.0, 9.0, 6.4)

crop_encoded = 1 if crop == "Wheat" else 0
temp_x_rainfall = avg_temp * rainfall
NPK_total = N + P + K

input_data = np.array([[year, rainfall, pesticides, avg_temp,
                        N, P, K, temperature, humidity, ph,
                        temp_x_rainfall, NPK_total, crop_encoded]])
input_df = pd.DataFrame(input_data, columns=FEATURES)

# Predictions
col1, col2, col3, col4 = st.columns(4)

rf_pred    = rf.predict(input_df)[0]
xgb_pred   = xgb.predict(input_df)[0]
input_sc   = scaler.transform(input_df)
svr_pred   = svr.predict(input_sc)[0]
stack_pred = stack.predict(input_df)[0]

col1.metric(" Random Forest", f"{rf_pred:.3f} T/Ha")
col2.metric(" XGBoost", f"{xgb_pred:.3f} T/Ha")
col3.metric(" SVR", f"{svr_pred:.3f} T/Ha")
col4.metric(" Ensemble", f"{stack_pred:.3f} T/Ha")

st.markdown("---")

# Model comparison chart
st.subheader(" Model Predictions Comparison")
fig, ax = plt.subplots(figsize=(8, 3))
models = ["Random Forest", "XGBoost", "SVR", "Ensemble"]
preds  = [rf_pred, xgb_pred, svr_pred, stack_pred]
colors = ["#2e8b57", "#e07b39", "#5b7fa6", "#8b2e8b"]
bars = ax.barh(models, preds, color=colors, edgecolor="white")
ax.set_xlabel("Predicted Yield (Ton/Ha)")
ax.set_title(f"Yield Predictions for {crop} — {year}")
for bar, val in zip(bars, preds):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontweight="bold")
plt.tight_layout()
st.pyplot(fig)

st.markdown("---")

# SHAP explanation
st.subheader(" SHAP Feature Importance (XGBoost)")
explainer = shap.TreeExplainer(xgb)
shap_vals = explainer.shap_values(input_df)
fig2, ax2 = plt.subplots(figsize=(8, 4))
shap.waterfall_plot(shap.Explanation(
    values=shap_vals[0],
    base_values=explainer.expected_value,
    data=input_df.iloc[0],
    feature_names=FEATURES), show=False)
plt.tight_layout()
st.pyplot(fig2)

st.markdown("---")
st.markdown("**Project:** Multi-Model Ensemble for Crop Yield Prediction | "
            "**Region:** Punjab, Pakistan | **Models:** RF + XGBoost + SVR + Stacking")
'''

with open(f'{PROJECT_DIR}/app.py', 'w') as f:
    f.write(app_code)

print(" Streamlit app saved to Drive!")
print(f" Location: {PROJECT_DIR}/app.py")

In [ ]:
# Check if app.py was saved correctly
import os
PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

if os.path.exists(f'{PROJECT_DIR}/app.py'):
    size = os.path.getsize(f'{PROJECT_DIR}/app.py')
    print(f" app.py exists! Size: {size} bytes")
    # Show first 5 lines
    with open(f'{PROJECT_DIR}/app.py', 'r') as f:
        lines = f.readlines()[:5]
    print("First 5 lines:")
    for line in lines:
        print(line, end='')
else:
    print(" app.py not found — will recreate it")

In [ ]:
# CELL 11 — Prepare files for Hugging Face upload

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

# Create requirements.txt
requirements = """streamlit
numpy
pandas
scikit-learn
xgboost
shap
matplotlib
joblib
"""

with open(f'{PROJECT_DIR}/requirements.txt', 'w') as f:
    f.write(requirements)
print(" requirements.txt saved!")

# List all files to upload
import os
print("\n Files to upload to Hugging Face:")
print(f"\n1. app.py")
print(f"2. requirements.txt")
print(f"\n3. models/ folder contents:")
for f in os.listdir(f'{PROJECT_DIR}/models'):
    size = os.path.getsize(f'{PROJECT_DIR}/models/{f}')
    print(f"   models/{f}  —  {size/1024:.1f} KB")
print(f"\n4. data/processed/final_ml_dataset.csv")

In [ ]:
# CELL 12 — Download all files to your PC

from google.colab import files
import os

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

# Download app.py
files.download(f'{PROJECT_DIR}/app.py')

# Download requirements.txt
files.download(f'{PROJECT_DIR}/requirements.txt')

# Download all models
for model_file in ['random_forest.pkl', 'xgboost.pkl', 'svr.pkl',
                   'scaler.pkl', 'stacking_ensemble.pkl']:
    files.download(f'{PROJECT_DIR}/models/{model_file}')

# Download dataset
files.download(f'{PROJECT_DIR}/data/processed/final_ml_dataset.csv')

print(" All files downloading to your PC!")
print("Check your Downloads folder")

In [ ]:
# CELL 13 — Create Gradio app for Hugging Face

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

app_code = '''
import gradio as gr
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

# Load models
rf     = joblib.load("models/random_forest.pkl")
xgb    = joblib.load("models/xgboost.pkl")
svr    = joblib.load("models/svr.pkl")
scaler = joblib.load("models/scaler.pkl")
stack  = joblib.load("models/stacking_ensemble.pkl")

FEATURES = ["year","rainfall_mm","pesticides_tonnes","avg_temp",
            "N","P","K","temperature","humidity","ph",
            "temp_x_rainfall","NPK_total","crop_encoded"]

def predict_yield(crop, year, rainfall, pesticides, avg_temp,
                  N, P, K, temperature, humidity, ph):

    crop_encoded   = 1 if crop == "Wheat" else 0
    temp_x_rainfall = avg_temp * rainfall
    NPK_total       = N + P + K

    input_data = np.array([[year, rainfall, pesticides, avg_temp,
                            N, P, K, temperature, humidity, ph,
                            temp_x_rainfall, NPK_total, crop_encoded]])
    input_df = pd.DataFrame(input_data, columns=FEATURES)
    input_sc = scaler.transform(input_df)

    rf_pred    = round(float(rf.predict(input_df)[0]), 3)
    xgb_pred   = round(float(xgb.predict(input_df)[0]), 3)
    svr_pred   = round(float(svr.predict(input_sc)[0]), 3)
    stack_pred = round(float(stack.predict(input_df)[0]), 3)

    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 3))
    models  = ["Random Forest", "XGBoost", "SVR", "Ensemble"]
    preds   = [rf_pred, xgb_pred, svr_pred, stack_pred]
    colors  = ["#2e8b57", "#e07b39", "#5b7fa6", "#8b2e8b"]
    bars = ax.barh(models, preds, color=colors, edgecolor="white")
    ax.set_xlabel("Predicted Yield (Ton/Ha)")
    ax.set_title(f"Yield Predictions for {crop} — {int(year)}")
    for bar, val in zip(bars, preds):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontweight="bold")
    plt.tight_layout()

    # SHAP waterfall
    explainer = shap.TreeExplainer(xgb)
    shap_vals = explainer.shap_values(input_df)
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_vals[0],
            base_values=explainer.expected_value,
            data=input_df.iloc[0],
            feature_names=FEATURES),
        show=False)
    plt.tight_layout()

    result = (
        f"🌲 Random Forest : {rf_pred} Ton/Ha\\n"
        f"⚡ XGBoost       : {xgb_pred} Ton/Ha\\n"
        f"🔵 SVR           : {svr_pred} Ton/Ha\\n"
        f"🏆 Ensemble      : {stack_pred} Ton/Ha"
    )

    return result, fig, fig2


# Build Gradio UI
with gr.Blocks(title="🌾 Crop Yield Predictor — Punjab Pakistan") as demo:

    gr.Markdown("# 🌾 Crop Yield Prediction — Punjab, Pakistan")
    gr.Markdown("### Multi-Model Ensemble: Random Forest + XGBoost + SVR + Stacking")
    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🌡️ Field Parameters")
            crop        = gr.Dropdown(["Wheat", "Maize"],
                                       value="Wheat", label="Crop Type")
            year        = gr.Slider(1990, 2025, value=2020,
                                    step=1, label="Year")
            rainfall    = gr.Slider(200, 1500, value=494,
                                    label="Rainfall (mm/year)")
            pesticides  = gr.Slider(0, 30000, value=6000,
                                    label="Pesticides (tonnes)")
            avg_temp    = gr.Slider(15.0, 35.0, value=24.0,
                                    label="Average Temperature (°C)")

        with gr.Column():
            gr.Markdown("### 🌱 Soil Parameters")
            N           = gr.Slider(0, 140, value=56,
                                    label="Nitrogen (N)")
            P           = gr.Slider(0, 100, value=42,
                                    label="Phosphorus (P)")
            K           = gr.Slider(0, 120, value=61,
                                    label="Potassium (K)")
            temperature = gr.Slider(15.0, 35.0, value=24.0,
                                    label="Soil Temperature (°C)")
            humidity    = gr.Slider(50.0, 100.0, value=86.0,
                                    label="Humidity (%)")
            ph          = gr.Slider(4.0, 9.0, value=6.4,
                                    label="Soil pH")

    predict_btn = gr.Button("🔍 Predict Yield", variant="primary")

    gr.Markdown("### 📊 Results")
    with gr.Row():
        result_text = gr.Textbox(label="Predicted Yields", lines=5)

    with gr.Row():
        chart1 = gr.Plot(label="Model Comparison Chart")
        chart2 = gr.Plot(label="SHAP Feature Importance")

    predict_btn.click(
        fn=predict_yield,
        inputs=[crop, year, rainfall, pesticides, avg_temp,
                N, P, K, temperature, humidity, ph],
        outputs=[result_text, chart1, chart2]
    )

    gr.Markdown("---")
    gr.Markdown("**Region:** Bahawalpur, Punjab Pakistan | "
                "**Dataset:** Kaggle Punjab Crop Data | "
                "**Models:** RF + XGBoost + SVR + Stacking Ensemble")

demo.launch()
'''

with open(f'{PROJECT_DIR}/app.py', 'w') as f:
    f.write(app_code)

# Update requirements.txt for Gradio
requirements = """gradio
numpy
pandas
scikit-learn
xgboost
shap
matplotlib
joblib
"""

with open(f'{PROJECT_DIR}/requirements.txt', 'w') as f:
    f.write(requirements)

print(" Gradio app.py saved!")
print(" requirements.txt updated!")

# Download both files
from google.colab import files
files.download(f'{PROJECT_DIR}/app.py')
files.download(f'{PROJECT_DIR}/requirements.txt')
print("✅ Both files downloading to your PC!")

In [ ]:
# CELL 14 — Fixed Gradio app

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

app_code = """
import gradio as gr
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

rf     = joblib.load("models/random_forest.pkl")
xgb    = joblib.load("models/xgboost.pkl")
svr    = joblib.load("models/svr.pkl")
scaler = joblib.load("models/scaler.pkl")
stack  = joblib.load("models/stacking_ensemble.pkl")

FEATURES = ["year","rainfall_mm","pesticides_tonnes","avg_temp",
            "N","P","K","temperature","humidity","ph",
            "temp_x_rainfall","NPK_total","crop_encoded"]

def predict_yield(crop, year, rainfall, pesticides, avg_temp,
                  N, P, K, temperature, humidity, ph):

    crop_encoded    = 1 if crop == "Wheat" else 0
    temp_x_rainfall = avg_temp * rainfall
    NPK_total       = N + P + K

    input_data = np.array([[year, rainfall, pesticides, avg_temp,
                            N, P, K, temperature, humidity, ph,
                            temp_x_rainfall, NPK_total, crop_encoded]])
    input_df = pd.DataFrame(input_data, columns=FEATURES)
    input_sc = scaler.transform(input_df)

    rf_pred    = round(float(rf.predict(input_df)[0]), 3)
    xgb_pred   = round(float(xgb.predict(input_df)[0]), 3)
    svr_pred   = round(float(svr.predict(input_sc)[0]), 3)
    stack_pred = round(float(stack.predict(input_df)[0]), 3)

    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 3))
    models_list = ["Random Forest", "XGBoost", "SVR", "Ensemble"]
    preds_list  = [rf_pred, xgb_pred, svr_pred, stack_pred]
    colors      = ["#2e8b57", "#e07b39", "#5b7fa6", "#8b2e8b"]
    bars = ax.barh(models_list, preds_list, color=colors, edgecolor="white")
    ax.set_xlabel("Predicted Yield (Ton/Ha)")
    ax.set_title("Yield Predictions for " + crop + " - " + str(int(year)))
    for bar, val in zip(bars, preds_list):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                str(val), va="center", fontweight="bold")
    plt.tight_layout()

    # SHAP waterfall
    explainer = shap.TreeExplainer(xgb)
    shap_vals = explainer.shap_values(input_df)
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_vals[0],
            base_values=explainer.expected_value,
            data=input_df.iloc[0],
            feature_names=FEATURES),
        show=False)
    plt.tight_layout()

    result = (
        "Random Forest : " + str(rf_pred) + " Ton/Ha" + chr(10) +
        "XGBoost       : " + str(xgb_pred) + " Ton/Ha" + chr(10) +
        "SVR           : " + str(svr_pred) + " Ton/Ha" + chr(10) +
        "Ensemble      : " + str(stack_pred) + " Ton/Ha"
    )

    return result, fig, fig2


with gr.Blocks(title="Crop Yield Predictor Punjab Pakistan") as demo:

    gr.Markdown("# Crop Yield Prediction — Punjab, Pakistan")
    gr.Markdown("### Multi-Model Ensemble: Random Forest + XGBoost + SVR + Stacking")
    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Field Parameters")
            crop        = gr.Dropdown(["Wheat", "Maize"],
                                      value="Wheat", label="Crop Type")
            year        = gr.Slider(1990, 2025, value=2020,
                                    step=1, label="Year")
            rainfall    = gr.Slider(200, 1500, value=494,
                                    label="Rainfall mm per year")
            pesticides  = gr.Slider(0, 30000, value=6000,
                                    label="Pesticides tonnes")
            avg_temp    = gr.Slider(15.0, 35.0, value=24.0,
                                    label="Average Temperature C")

        with gr.Column():
            gr.Markdown("### Soil Parameters")
            N           = gr.Slider(0, 140, value=56, label="Nitrogen N")
            P           = gr.Slider(0, 100, value=42, label="Phosphorus P")
            K           = gr.Slider(0, 120, value=61, label="Potassium K")
            temperature = gr.Slider(15.0, 35.0, value=24.0,
                                    label="Soil Temperature C")
            humidity    = gr.Slider(50.0, 100.0, value=86.0,
                                    label="Humidity percent")
            ph          = gr.Slider(4.0, 9.0, value=6.4, label="Soil pH")

    predict_btn = gr.Button("Predict Yield", variant="primary")

    gr.Markdown("### Results")
    with gr.Row():
        result_text = gr.Textbox(label="Predicted Yields", lines=5)

    with gr.Row():
        chart1 = gr.Plot(label="Model Comparison Chart")
        chart2 = gr.Plot(label="SHAP Feature Importance")

    predict_btn.click(
        fn=predict_yield,
        inputs=[crop, year, rainfall, pesticides, avg_temp,
                N, P, K, temperature, humidity, ph],
        outputs=[result_text, chart1, chart2]
    )

    gr.Markdown("---")
    gr.Markdown("Region: Bahawalpur, Punjab Pakistan | Dataset: Kaggle Punjab Crop Data | Models: RF + XGBoost + SVR + Stacking Ensemble")

demo.launch()
"""

with open(f'{PROJECT_DIR}/app.py', 'w') as f:
    f.write(app_code)

print(" Fixed app.py saved!")

from google.colab import files
files.download(f'{PROJECT_DIR}/app.py')
print(" Downloading now!")

In [ ]:
# CELL 16 — Fix app.py to load models from root folder

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

app_code = """
import gradio as gr
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import os

# Load models from same folder as app.py
BASE = os.path.dirname(os.path.abspath(__file__))

rf     = joblib.load(os.path.join(BASE, "random_forest.pkl"))
xgb    = joblib.load(os.path.join(BASE, "xgboost.pkl"))
svr    = joblib.load(os.path.join(BASE, "svr.pkl"))
scaler = joblib.load(os.path.join(BASE, "scaler.pkl"))
stack  = joblib.load(os.path.join(BASE, "stacking_ensemble.pkl"))

FEATURES = ["year","rainfall_mm","pesticides_tonnes","avg_temp",
            "N","P","K","temperature","humidity","ph",
            "temp_x_rainfall","NPK_total","crop_encoded"]

def predict_yield(crop, year, rainfall, pesticides, avg_temp,
                  N, P, K, temperature, humidity, ph):

    crop_encoded    = 1 if crop == "Wheat" else 0
    temp_x_rainfall = avg_temp * rainfall
    NPK_total       = N + P + K

    input_data = np.array([[year, rainfall, pesticides, avg_temp,
                            N, P, K, temperature, humidity, ph,
                            temp_x_rainfall, NPK_total, crop_encoded]])
    input_df = pd.DataFrame(input_data, columns=FEATURES)
    input_sc = scaler.transform(input_df)

    rf_pred    = round(float(rf.predict(input_df)[0]), 3)
    xgb_pred   = round(float(xgb.predict(input_df)[0]), 3)
    svr_pred   = round(float(svr.predict(input_sc)[0]), 3)
    stack_pred = round(float(stack.predict(input_df)[0]), 3)

    fig, ax = plt.subplots(figsize=(8, 3))
    models_list = ["Random Forest", "XGBoost", "SVR", "Ensemble"]
    preds_list  = [rf_pred, xgb_pred, svr_pred, stack_pred]
    colors      = ["#2e8b57", "#e07b39", "#5b7fa6", "#8b2e8b"]
    bars = ax.barh(models_list, preds_list, color=colors, edgecolor="white")
    ax.set_xlabel("Predicted Yield (Ton/Ha)")
    ax.set_title("Yield Predictions for " + crop + " - " + str(int(year)))
    for bar, val in zip(bars, preds_list):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                str(val), va="center", fontweight="bold")
    plt.tight_layout()

    explainer = shap.TreeExplainer(xgb)
    shap_vals = explainer.shap_values(input_df)
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_vals[0],
            base_values=explainer.expected_value,
            data=input_df.iloc[0],
            feature_names=FEATURES),
        show=False)
    plt.tight_layout()

    result = (
        "Random Forest : " + str(rf_pred) + " Ton/Ha" + chr(10) +
        "XGBoost       : " + str(xgb_pred) + " Ton/Ha" + chr(10) +
        "SVR           : " + str(svr_pred) + " Ton/Ha" + chr(10) +
        "Ensemble      : " + str(stack_pred) + " Ton/Ha"
    )

    return result, fig, fig2


with gr.Blocks(title="Crop Yield Predictor Punjab Pakistan") as demo:

    gr.Markdown("# Crop Yield Prediction — Punjab, Pakistan")
    gr.Markdown("### Multi-Model Ensemble: Random Forest + XGBoost + SVR + Stacking")
    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Field Parameters")
            crop        = gr.Dropdown(["Wheat", "Maize"],
                                      value="Wheat", label="Crop Type")
            year        = gr.Slider(1990, 2025, value=2020,
                                    step=1, label="Year")
            rainfall    = gr.Slider(200, 1500, value=494,
                                    label="Rainfall mm per year")
            pesticides  = gr.Slider(0, 30000, value=6000,
                                    label="Pesticides tonnes")
            avg_temp    = gr.Slider(15.0, 35.0, value=24.0,
                                    label="Average Temperature C")

        with gr.Column():
            gr.Markdown("### Soil Parameters")
            N           = gr.Slider(0, 140, value=56, label="Nitrogen N")
            P           = gr.Slider(0, 100, value=42, label="Phosphorus P")
            K           = gr.Slider(0, 120, value=61, label="Potassium K")
            temperature = gr.Slider(15.0, 35.0, value=24.0,
                                    label="Soil Temperature C")
            humidity    = gr.Slider(50.0, 100.0, value=86.0,
                                    label="Humidity percent")
            ph          = gr.Slider(4.0, 9.0, value=6.4, label="Soil pH")

    predict_btn = gr.Button("Predict Yield", variant="primary")

    gr.Markdown("### Results")
    with gr.Row():
        result_text = gr.Textbox(label="Predicted Yields", lines=5)

    with gr.Row():
        chart1 = gr.Plot(label="Model Comparison Chart")
        chart2 = gr.Plot(label="SHAP Feature Importance")

    predict_btn.click(
        fn=predict_yield,
        inputs=[crop, year, rainfall, pesticides, avg_temp,
                N, P, K, temperature, humidity, ph],
        outputs=[result_text, chart1, chart2]
    )

    gr.Markdown("---")
    gr.Markdown("Region: Bahawalpur, Punjab Pakistan | Models: RF + XGBoost + SVR + Stacking Ensemble")

demo.launch()
"""

with open(f'{PROJECT_DIR}/app.py', 'w') as f:
    f.write(app_code)

print(" Fixed app.py saved!")

from google.colab import files
files.download(f'{PROJECT_DIR}/app.py')
print(" Downloading!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib
import warnings
warnings.filterwarnings('ignore')
import os

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'
processed  = f'{PROJECT_DIR}/data/processed'
outputs    = f'{PROJECT_DIR}/outputs'
models_dir = f'{PROJECT_DIR}/models'

# ── Load data ──
file_path = f'{processed}/final_ml_dataset.csv'
if not os.path.exists(file_path):
    print(f"Error: File not found at {file_path}. Please ensure Cell 6 ran successfully and the file exists.")
    # You might want to halt execution or re-run Cell 6 here if this were an interactive session
else:
    df = pd.read_csv(file_path)

    # Drop crop_encoded — use only real environmental/soil features
    FEATURES = ['year', 'rainfall_mm', 'pesticides_tonnes', 'avg_temp',
                'N', 'P', 'K', 'temperature', 'humidity', 'ph',
                'temp_x_rainfall', 'NPK_total']
    TARGET = 'yield_ton_ha'

    X = df[FEATURES]
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)
    print(f"Train: {X_train.shape} | Test: {X_test.shape}")
    print(f"Features used: {FEATURES}\n")

    # Scale for SVR
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # ── Helper ──
    def evaluate(name, model, X_tr, X_te, y_tr, y_te):
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        mae  = mean_absolute_error(y_te, pred)
        rmse = np.sqrt(mean_squared_error(y_te, pred))
        r2   = r2_score(y_te, pred)
        cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()
        print(f"{'─'*45}")
        print(f"  {name}")
        print(f"  MAE  : {mae:.4f}")
        print(f"  RMSE : {rmse:.4f}")
        print(f"  R²   : {r2:.4f}")
        print(f"  CV R²: {cv:.4f}")
        return {'Model': name, 'MAE': round(mae,4),
                'RMSE': round(rmse,4), 'R²': round(r2,4),
                'CV R²': round(cv,4)}, pred

    # ── MODEL 1: Random Forest ──
    rf = RandomForestRegressor(n_estimators=200, max_depth=10,
                               random_state=42, n_jobs=-1)
    rf_res, rf_pred = evaluate('Random Forest', rf,
                                X_train, X_test, y_train, y_test)
    joblib.dump(rf, f'{models_dir}/random_forest_v2.pkl')

    # ── MODEL 2: XGBoost ──
    xgb = XGBRegressor(n_estimators=200, learning_rate=0.05,
                       max_depth=6, random_state=42, verbosity=0)
    xgb_res, xgb_pred = evaluate('XGBoost', xgb,
                                   X_train, X_test, y_train, y_test)
    joblib.dump(xgb, f'{models_dir}/xgboost_v2.pkl')

    # ── MODEL 3: SVR ──
    svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
    svr_res, svr_pred = evaluate('SVR', svr,
                                  X_train_sc, X_test_sc, y_train, y_test)
    joblib.dump(svr, f'{models_dir}/svr_v2.pkl')
    joblib.dump(scaler, f'{models_dir}/scaler_v2.pkl')

    # ── STACKING ENSEMBLE ──
    svr_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1))
    ])
    base_learners = [
        ('rf',  RandomForestRegressor(n_estimators=200, max_depth=10,
                                       random_state=42, n_jobs=-1)),
        ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05,
                              max_depth=6, random_state=42, verbosity=0)),
        ('svr', svr_pipe)
    ]
    stack = StackingRegressor(estimators=base_learners,
                              final_estimator=Ridge(alpha=1.0),
                              cv=5, n_jobs=-1)
    stack.fit(X_train, y_train)
    stack_pred = stack.predict(X_test)
    mae  = mean_absolute_error(y_test, stack_pred)
    rmse = np.sqrt(mean_squared_error(y_test, stack_pred))
    r2   = r2_score(y_test, stack_pred)
    cv   = cross_val_score(stack, X_train, y_train, cv=5, scoring='r2').mean()
    stack_res = {'Model': 'Stacking Ensemble', 'MAE': round(mae,4),
                 'RMSE': round(rmse,4), 'R²': round(r2,4), 'CV R²': round(cv,4)}
    print(f"{'─'*45}")
    print(f"  Stacking Ensemble")
    print(f"  MAE  : {mae:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  R²   : {r2:.4f}")
    print(f"  CV R²: {cv:.4f}")
    joblib.dump(stack, f'{models_dir}/stacking_ensemble_v2.pkl')

    # ── FINAL TABLE ──
    results = pd.DataFrame([rf_res, xgb_res, svr_res, stack_res])
    print(f"\n{'='*55}")
    print(" FINAL MODEL COMPARISON (without crop_encoded)")
    print('='*55)
    print(results.to_string(index=False))
    results.to_csv(f'{outputs}/model_results_v2.csv', index=False)

    # ── COMPARISON BAR CHART ──
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    metrics = ['MAE', 'RMSE', 'R²']
    colors  = ['#2e8b57', '#e07b39', '#5b7fa6', '#8b2e8b']
    for i, metric in enumerate(metrics):
        axes[i].bar(results['Model'], results[metric],
                    color=colors, edgecolor='white', width=0.5)
        axes[i].set_title(metric, fontsize=13, fontweight='bold')
        axes[i].set_ylabel(metric)
        axes[i].set_ylim(0, results[metric].max() * 1.3)
        for j, v in enumerate(results[metric]):
            axes[i].text(j, v + 0.005, str(v),
                         ha='center', fontsize=9, fontweight='bold')
        axes[i].tick_params(axis='x', rotation=20)
    plt.suptitle('Model Comparison — Crop Yield Prediction (No Crop Label)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{outputs}/model_comparison_v2.png', dpi=150)
    plt.show()
    print(" Chart saved!")

    # ── ACTUAL vs PREDICTED ──
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    preds = [rf_pred, xgb_pred, svr_pred]
    names = ['Random Forest', 'XGBoost', 'SVR']
    cols  = ['#2e8b57', '#e07b39', '#5b7fa6']
    for i, (pred, name, col) in enumerate(zip(preds, names, cols)):
        axes[i].scatter(y_test, pred, alpha=0.6, color=col,
                        edgecolors='white', s=60)
        mn = min(y_test.min(), pred.min())
        mx = max(y_test.max(), pred.max())
        axes[i].plot([mn, mx], [mn, mx], 'k--', linewidth=1.5)
        axes[i].set_title(name, fontweight='bold')
        axes[i].set_xlabel('Actual Yield')
        axes[i].set_ylabel('Predicted Yield')
    plt.suptitle('Actual vs Predicted — Without Crop Label Feature',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{outputs}/actual_vs_predicted_v2.png', dpi=150)
    plt.show()
    print(" All saved!")
    print("\n Retrain complete! Paste results here.")


In [ ]:
# CELL 18 — Final retrain: only real soil + climate features

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib, warnings
warnings.filterwarnings('ignore')

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'
processed  = f'{PROJECT_DIR}/data/processed'
outputs    = f'{PROJECT_DIR}/outputs'
models_dir = f'{PROJECT_DIR}/models'

df = pd.read_csv(f'{processed}/final_ml_dataset.csv')

# Only genuine variable features — no year, no constant rainfall
FEATURES = ['pesticides_tonnes', 'avg_temp', 'N', 'P', 'K',
            'temperature', 'humidity', 'ph', 'NPK_total']
TARGET = 'yield_ton_ha'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features: {FEATURES}\n")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    r2   = r2_score(y_te, pred)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()
    print(f"{'─'*45}")
    print(f"  {name}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  R²   : {r2:.4f}")
    print(f"  CV R²: {cv:.4f}")
    return {'Model': name, 'MAE': round(mae,4),
            'RMSE': round(rmse,4), 'R²': round(r2,4),
            'CV R²': round(cv,4)}, pred

# MODEL 1: Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=8,
                           random_state=42, n_jobs=-1)
rf_res, rf_pred = evaluate('Random Forest', rf,
                            X_train, X_test, y_train, y_test)
joblib.dump(rf, f'{models_dir}/rf_final.pkl')

# MODEL 2: XGBoost
xgb = XGBRegressor(n_estimators=200, learning_rate=0.05,
                   max_depth=5, random_state=42, verbosity=0)
xgb_res, xgb_pred = evaluate('XGBoost', xgb,
                               X_train, X_test, y_train, y_test)
joblib.dump(xgb, f'{models_dir}/xgb_final.pkl')

# MODEL 3: SVR
svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
svr_res, svr_pred = evaluate('SVR', svr,
                              X_train_sc, X_test_sc, y_train, y_test)
joblib.dump(svr, f'{models_dir}/svr_final.pkl')
joblib.dump(scaler, f'{models_dir}/scaler_final.pkl')

# STACKING ENSEMBLE
svr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1))
])
stack = StackingRegressor(
    estimators=[
        ('rf',  RandomForestRegressor(n_estimators=200, max_depth=8,
                                       random_state=42, n_jobs=-1)),
        ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05,
                              max_depth=5, random_state=42, verbosity=0)),
        ('svr', svr_pipe)
    ],
    final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1
)
stack.fit(X_train, y_train)
stack_pred = stack.predict(X_test)
mae  = mean_absolute_error(y_test, stack_pred)
rmse = np.sqrt(mean_squared_error(y_test, stack_pred))
r2   = r2_score(y_test, stack_pred)
cv   = cross_val_score(stack, X_train, y_train, cv=5, scoring='r2').mean()
stack_res = {'Model':'Stacking Ensemble','MAE':round(mae,4),
             'RMSE':round(rmse,4),'R²':round(r2,4),'CV R²':round(cv,4)}
print(f"{'─'*45}")
print(f"  Stacking Ensemble")
print(f"  MAE  : {mae:.4f}")
print(f"  RMSE : {rmse:.4f}")
print(f"  R²   : {r2:.4f}")
print(f"  CV R²: {cv:.4f}")
joblib.dump(stack, f'{models_dir}/stack_final.pkl')

# FINAL TABLE
results = pd.DataFrame([rf_res, xgb_res, svr_res, stack_res])
print(f"\n{'='*55}")
print(" FINAL RESULTS (genuine features only)")
print('='*55)
print(results.to_string(index=False))
results.to_csv(f'{outputs}/model_results_final.csv', index=False)

# BAR CHART
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['#2e8b57','#e07b39','#5b7fa6','#8b2e8b']
for i, metric in enumerate(['MAE','RMSE','R²']):
    axes[i].bar(results['Model'], results[metric],
                color=colors, edgecolor='white', width=0.5)
    axes[i].set_title(metric, fontsize=13, fontweight='bold')
    axes[i].set_ylim(0, results[metric].max() * 1.3)
    for j, v in enumerate(results[metric]):
        axes[i].text(j, v+0.005, str(v), ha='center',
                     fontsize=9, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20)
plt.suptitle('Model Comparison — Genuine Features Only (Punjab Crop Yield)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{outputs}/model_comparison_final.png', dpi=150)
plt.show()

# ACTUAL vs PREDICTED
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (pred, name, col) in enumerate(zip(
        [rf_pred, xgb_pred, svr_pred],
        ['Random Forest','XGBoost','SVR'],
        ['#2e8b57','#e07b39','#5b7fa6'])):
    axes[i].scatter(y_test, pred, alpha=0.6, color=col,
                    edgecolors='white', s=60)
    mn = min(y_test.min(), pred.min())
    mx = max(y_test.max(), pred.max())
    axes[i].plot([mn,mx],[mn,mx],'k--',linewidth=1.5)
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel('Actual Yield')
    axes[i].set_ylabel('Predicted Yield')
plt.suptitle('Actual vs Predicted — Genuine Features Only',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{outputs}/actual_vs_predicted_final.png', dpi=150)
plt.show()
print("\n Done! Paste full output here.")

In [ ]:
# CELL 19 — Download final models for Hugging Face update

from google.colab import files

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

# Download all final models
files.download(f'{PROJECT_DIR}/models/rf_final.pkl')
files.download(f'{PROJECT_DIR}/models/xgb_final.pkl')
files.download(f'{PROJECT_DIR}/models/svr_final.pkl')
files.download(f'{PROJECT_DIR}/models/scaler_final.pkl')
files.download(f'{PROJECT_DIR}/models/stack_final.pkl')

# Save updated app.py for Hugging Face
app_code = """
import gradio as gr
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import os

BASE = os.path.dirname(os.path.abspath(__file__))

rf    = joblib.load(os.path.join(BASE, "rf_final.pkl"))
xgb   = joblib.load(os.path.join(BASE, "xgb_final.pkl"))
svr   = joblib.load(os.path.join(BASE, "svr_final.pkl"))
scaler= joblib.load(os.path.join(BASE, "scaler_final.pkl"))
stack = joblib.load(os.path.join(BASE, "stack_final.pkl"))

FEATURES = ['pesticides_tonnes', 'avg_temp', 'N', 'P', 'K',
            'temperature', 'humidity', 'ph', 'NPK_total']

def predict_yield(pesticides, avg_temp, N, P, K,
                  temperature, humidity, ph):

    NPK_total = N + P + K
    input_data = np.array([[pesticides, avg_temp, N, P, K,
                            temperature, humidity, ph, NPK_total]])
    input_df = pd.DataFrame(input_data, columns=FEATURES)
    input_sc = scaler.transform(input_df)

    rf_pred    = round(float(rf.predict(input_df)[0]), 3)
    xgb_pred   = round(float(xgb.predict(input_df)[0]), 3)
    svr_pred   = round(float(svr.predict(input_sc)[0]), 3)
    stack_pred = round(float(stack.predict(input_df)[0]), 3)

    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 3))
    models_list = ["Random Forest", "XGBoost", "SVR", "Ensemble"]
    preds_list  = [rf_pred, xgb_pred, svr_pred, stack_pred]
    colors      = ["#2e8b57", "#e07b39", "#5b7fa6", "#8b2e8b"]
    bars = ax.barh(models_list, preds_list, color=colors, edgecolor="white")
    ax.set_xlabel("Predicted Yield (Ton/Ha)")
    ax.set_title("Crop Yield Predictions — Punjab, Pakistan")
    for bar, val in zip(bars, preds_list):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                str(val), va="center", fontweight="bold")
    plt.tight_layout()

    # SHAP
    explainer = shap.TreeExplainer(xgb)
    shap_vals = explainer.shap_values(input_df)
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_vals[0],
            base_values=explainer.expected_value,
            data=input_df.iloc[0],
            feature_names=FEATURES),
        show=False)
    plt.tight_layout()

    result = (
        "Random Forest : " + str(rf_pred) + " Ton/Ha" + chr(10) +
        "XGBoost       : " + str(xgb_pred) + " Ton/Ha" + chr(10) +
        "SVR           : " + str(svr_pred) + " Ton/Ha" + chr(10) +
        "Ensemble      : " + str(stack_pred) + " Ton/Ha"
    )
    return result, fig, fig2


with gr.Blocks(title="Crop Yield Predictor — Punjab Pakistan") as demo:

    gr.Markdown("# Crop Yield Prediction — Punjab, Pakistan")
    gr.Markdown("### Multi-Model Ensemble: Random Forest + XGBoost + SVR + Stacking")
    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Climate Parameters")
            pesticides  = gr.Slider(0, 30000, value=6000,
                                    label="Pesticides (tonnes)")
            avg_temp    = gr.Slider(15.0, 35.0, value=24.0,
                                    label="Average Temperature (C)")

        with gr.Column():
            gr.Markdown("### Soil Parameters")
            N           = gr.Slider(0, 140, value=56, label="Nitrogen (N)")
            P           = gr.Slider(0, 100, value=42, label="Phosphorus (P)")
            K           = gr.Slider(0, 120, value=61, label="Potassium (K)")
            temperature = gr.Slider(15.0, 35.0, value=24.0,
                                    label="Soil Temperature (C)")
            humidity    = gr.Slider(50.0, 100.0, value=86.0,
                                    label="Humidity (%)")
            ph          = gr.Slider(4.0, 9.0, value=6.4, label="Soil pH")

    predict_btn = gr.Button("Predict Yield", variant="primary")

    gr.Markdown("### Results")
    with gr.Row():
        result_text = gr.Textbox(label="Predicted Yields", lines=5)
    with gr.Row():
        chart1 = gr.Plot(label="Model Comparison")
        chart2 = gr.Plot(label="SHAP Feature Importance")

    predict_btn.click(
        fn=predict_yield,
        inputs=[pesticides, avg_temp, N, P, K,
                temperature, humidity, ph],
        outputs=[result_text, chart1, chart2]
    )

    gr.Markdown("---")
    gr.Markdown("Region: Bahawalpur, Punjab Pakistan | "
                "Models: RF (R²=0.97) + XGBoost (R²=0.99) + "
                "SVR (R²=0.22) + Stacking (R²=0.99)")

demo.launch()
"""

import os
with open(f'{PROJECT_DIR}/app_final.py', 'w') as f:
    f.write(app_code)
files.download(f'{PROJECT_DIR}/app_final.py')

print(" All files downloading!")
print("Upload to Hugging Face:")
print("  - rf_final.pkl")
print("  - xgb_final.pkl")
print("  - svr_final.pkl")
print("  - scaler_final.pkl")
print("  - stack_final.pkl")
print("  - app_final.py  (rename to app.py when uploading)")

In [ ]:
# Save 9-feature dataset for Orange
import pandas as pd
PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

df = pd.read_csv(f'{PROJECT_DIR}/data/processed/final_ml_dataset.csv')

FEATURES = ['pesticides_tonnes', 'avg_temp', 'N', 'P', 'K',
            'temperature', 'humidity', 'ph', 'NPK_total', 'yield_ton_ha']

df_orange = df[FEATURES]
df_orange.to_csv(f'{PROJECT_DIR}/data/processed/orange_final.csv', index=False)
print(df_orange.shape)
print(df_orange.columns.tolist())

from google.colab import files
files.download(f'{PROJECT_DIR}/data/processed/orange_final.csv')
print("Downloaded!")

In [ ]:
# CELL 20A — Install dependencies + Download PlantVillage dataset

!pip install kaggle -q

import os
from google.colab import files

# Make sure Kaggle is ready
os.makedirs('/root/.config/kaggle', exist_ok=True)
!cp /content/drive/MyDrive/CropYieldProject/kaggle.json /root/.config/kaggle/kaggle.json 2>/dev/null || \
 cp /root/.config/kaggle/kaggle.json /root/.config/kaggle/kaggle.json 2>/dev/null
!chmod 600 /root/.config/kaggle/kaggle.json

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'
CV_DIR = f'{PROJECT_DIR}/cv_module'
os.makedirs(f'{CV_DIR}/data', exist_ok=True)
os.makedirs(f'{CV_DIR}/model', exist_ok=True)

# Download PlantVillage dataset (wheat + maize leaf images)
!kaggle datasets download -d abdallahalidev/plantvillage-dataset \
    -p {CV_DIR}/data --unzip

print(" Done! Let's see what downloaded:")
import os
for folder in os.listdir(f'{CV_DIR}/data'):
    print(f"   {folder}")

In [ ]:
# CELL 20B — Explore PlantVillage structure

import os

CV_DIR = '/content/drive/MyDrive/CropYieldProject/cv_module'

# Find the dataset folder
base = f'{CV_DIR}/data/plantvillage dataset'

# List top-level folders
for folder in sorted(os.listdir(base)):
    print(f" {folder}")

In [ ]:
# CELL 20C — Check wheat and maize classes available

import os

color_dir = '/content/drive/MyDrive/CropYieldProject/cv_module/data/plantvillage dataset/color'

all_classes = sorted(os.listdir(color_dir))
print(f"Total classes: {len(all_classes)}\n")

# Show all classes
for c in all_classes:
    count = len(os.listdir(os.path.join(color_dir, c)))
    print(f"  {c}  —  {count} images")

In [ ]:
# CELL 20D — Prepare maize leaf dataset (healthy vs diseased)

import os, shutil
import numpy as np
from sklearn.model_selection import train_test_split

CV_DIR   = '/content/drive/MyDrive/CropYieldProject/cv_module'
color_dir = f'{CV_DIR}/data/plantvillage dataset/color'
out_dir   = f'{CV_DIR}/data/prepared'

# Clean and recreate
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
os.makedirs(f'{out_dir}/train/healthy', exist_ok=True)
os.makedirs(f'{out_dir}/train/diseased', exist_ok=True)
os.makedirs(f'{out_dir}/val/healthy',   exist_ok=True)
os.makedirs(f'{out_dir}/val/diseased',  exist_ok=True)

def copy_images(src_folder, label, split=0.2):
    imgs = [f for f in os.listdir(src_folder)
            if f.lower().endswith(('.jpg','.jpeg','.png'))]
    train_imgs, val_imgs = train_test_split(imgs, test_size=split, random_state=42)
    for f in train_imgs:
        shutil.copy(os.path.join(src_folder, f),
                    os.path.join(out_dir, 'train', label, f))
    for f in val_imgs:
        shutil.copy(os.path.join(src_folder, f),
                    os.path.join(out_dir, 'val', label, f))
    print(f"  {label} | {src_folder.split('/')[-1][:40]} → train:{len(train_imgs)} val:{len(val_imgs)}")

# Healthy
copy_images(f'{color_dir}/Corn_(maize)___healthy', 'healthy')

# Diseased (3 classes combined)
copy_images(f'{color_dir}/Corn_(maize)___Common_rust_', 'diseased')
copy_images(f'{color_dir}/Corn_(maize)___Northern_Leaf_Blight', 'diseased')
copy_images(f'{color_dir}/Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'diseased')

# Summary
for split in ['train', 'val']:
    for label in ['healthy', 'diseased']:
        n = len(os.listdir(f'{out_dir}/{split}/{label}'))
        print(f"  {split}/{label}: {n} images")

print("\n Dataset prepared!")

In [ ]:
# CELL 20E — Train MobileNetV2 leaf health classifier

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

CV_DIR    = '/content/drive/MyDrive/CropYieldProject/cv_module'
data_dir  = f'{CV_DIR}/data/prepared'
model_dir = f'{CV_DIR}/model'

IMG_SIZE  = 224
BATCH     = 32

# ── Data generators ──
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)
val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    f'{data_dir}/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='binary',
    classes=['diseased', 'healthy']   # diseased=0, healthy=1
)
val_data = val_gen.flow_from_directory(
    f'{data_dir}/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='binary',
    classes=['diseased', 'healthy']
)

print(f"Class indices: {train_data.class_indices}")

# ── Build MobileNetV2 ──
base = MobileNetV2(weights='imagenet',
                   include_top=False,
                   input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False   # freeze base layers

x = base.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
out = Dense(1, activation='sigmoid')(x)   # 1 = healthy, 0 = diseased

model = Model(inputs=base.input, outputs=out)
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

print(f"\nTrainable params: {model.count_params():,}")
print(f"Train batches: {len(train_data)} | Val batches: {len(val_data)}")

# ── Train ──
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(f'{model_dir}/leaf_classifier.h5',
                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

# ── Plot training ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],    label='Train', color='#2e8b57')
axes[0].plot(history.history['val_accuracy'],label='Val',   color='#e07b39')
axes[0].set_title('Accuracy', fontweight='bold')
axes[0].legend()

axes[1].plot(history.history['loss'],        label='Train', color='#2e8b57')
axes[1].plot(history.history['val_loss'],    label='Val',   color='#e07b39')
axes[1].set_title('Loss', fontweight='bold')
axes[1].legend()

plt.suptitle('MobileNetV2 — Leaf Health Classifier Training',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CV_DIR}/model/training_curves.png', dpi=150)
plt.show()

print(f"\n Model saved to {model_dir}/leaf_classifier.h5")
print(f"Final val accuracy: {max(history.history['val_accuracy']):.4f}")

In [ ]:
# CELL 20F — Evaluate CV model + generate health_score for dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

CV_DIR    = '/content/drive/MyDrive/CropYieldProject/cv_module'
PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

# Load model
model = load_model(f'{CV_DIR}/model/leaf_classifier.h5')
print(" Model loaded!")

# Evaluate on val set
val_gen = ImageDataGenerator(rescale=1./255)
val_data = val_gen.flow_from_directory(
    f'{CV_DIR}/data/prepared/val',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    classes=['diseased', 'healthy'],
    shuffle=False
)

preds = (model.predict(val_data) > 0.5).astype(int).flatten()
true  = val_data.classes

# Classification report
print("\n📊 Classification Report:")
print(classification_report(true, preds,
      target_names=['diseased', 'healthy']))

# Confusion matrix
cm = confusion_matrix(true, preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['diseased','healthy'],
            yticklabels=['diseased','healthy'])
plt.title('Confusion Matrix — Leaf Health Classifier', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{CV_DIR}/model/confusion_matrix.png', dpi=150)
plt.show()

# ── Generate health_score for full ML dataset ──
# health_score = probability of being healthy (0=diseased, 1=healthy)
# We'll assign based on crop type distribution from our trained model
df = pd.read_csv(f'{PROJECT_DIR}/data/processed/final_ml_dataset.csv')

# Get average predicted probabilities per class from val set
raw_preds = model.predict(val_data).flatten()
avg_healthy_score  = float(raw_preds[true == 1].mean())
avg_diseased_score = float(raw_preds[true == 0].mean())

print(f"\nAvg health_score for healthy leaves : {avg_healthy_score:.4f}")
print(f"Avg health_score for diseased leaves: {avg_diseased_score:.4f}")

# Assign health_score to dataset rows using NPK_total as proxy
# (higher NPK → healthier crop → higher health score)
# Normalize NPK_total to 0-1 range and blend with model scores
npk = df['NPK_total'].values
npk_norm = (npk - npk.min()) / (npk.max() - npk.min())

# health_score: NPK-based continuous score scaled between diseased and healthy range
df['health_score'] = avg_diseased_score + npk_norm * (avg_healthy_score - avg_diseased_score)
df['health_score'] = df['health_score'].round(4)

print(f"\n health_score added to dataset!")
print(f"  Range: {df['health_score'].min():.4f} — {df['health_score'].max():.4f}")
print(f"  Mean : {df['health_score'].mean():.4f}")
print(f"\nSample rows:")
print(df[['pesticides_tonnes','NPK_total','health_score','yield_ton_ha']].head(8))

# Save updated dataset
df.to_csv(f'{PROJECT_DIR}/data/processed/final_ml_dataset_cv.csv', index=False)
print(f"\n Saved: final_ml_dataset_cv.csv  shape: {df.shape}")

In [ ]:
# CELL 20G — Check dataset + retrain ensemble with health_score as 10th feature

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib, warnings
warnings.filterwarnings('ignore')

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'
models_dir  = f'{PROJECT_DIR}/models'
outputs     = f'{PROJECT_DIR}/outputs'

# Load and inspect
df = pd.read_csv(f'{PROJECT_DIR}/data/processed/final_ml_dataset_cv.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nhealth_score unique values: {df['health_score'].nunique()}")
print(f"health_score range: {df['health_score'].min():.4f} — {df['health_score'].max():.4f}")

# ── Final 10 features (9 original + health_score) ──
FEATURES = ['pesticides_tonnes', 'avg_temp', 'N', 'P', 'K',
            'temperature', 'humidity', 'ph', 'NPK_total', 'health_score']
TARGET = 'yield_ton_ha'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f"\nTrain: {X_train.shape} | Test: {X_test.shape}")
print(f"Features: {FEATURES}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    r2   = r2_score(y_te, pred)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()
    print(f"{'─'*45}")
    print(f"  {name}")
    print(f"  MAE  : {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f} | CV R²: {cv:.4f}")
    return {'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4),
            'R²': round(r2,4), 'CV R²': round(cv,4)}, pred

# Train all 3 + stacking
rf  = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_res, rf_pred = evaluate('Random Forest', rf, X_train, X_test, y_train, y_test)
joblib.dump(rf, f'{models_dir}/rf_cv.pkl')

xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5,
                   random_state=42, verbosity=0)
xgb_res, xgb_pred = evaluate('XGBoost', xgb, X_train, X_test, y_train, y_test)
joblib.dump(xgb, f'{models_dir}/xgb_cv.pkl')

svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
svr_res, svr_pred = evaluate('SVR', svr, X_train_sc, X_test_sc, y_train, y_test)
joblib.dump(svr, f'{models_dir}/svr_cv.pkl')
joblib.dump(scaler, f'{models_dir}/scaler_cv.pkl')

svr_pipe = Pipeline([('scaler', StandardScaler()),
                     ('svr', SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1))])
stack = StackingRegressor(
    estimators=[
        ('rf',  RandomForestRegressor(n_estimators=200, max_depth=8,
                                      random_state=42, n_jobs=-1)),
        ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05,
                             max_depth=5, random_state=42, verbosity=0)),
        ('svr', svr_pipe)
    ],
    final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1
)
stack.fit(X_train, y_train)
stack_pred = stack.predict(X_test)
mae  = mean_absolute_error(y_test, stack_pred)
rmse = np.sqrt(mean_squared_error(y_test, stack_pred))
r2   = r2_score(y_test, stack_pred)
cv   = cross_val_score(stack, X_train, y_train, cv=5, scoring='r2').mean()
stack_res = {'Model':'Stacking Ensemble','MAE':round(mae,4),
             'RMSE':round(rmse,4),'R²':round(r2,4),'CV R²':round(cv,4)}
print(f"{'─'*45}")
print(f"  Stacking Ensemble")
print(f"  MAE  : {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f} | CV R²: {cv:.4f}")
joblib.dump(stack, f'{models_dir}/stack_cv.pkl')

# Comparison table
results = pd.DataFrame([rf_res, xgb_res, svr_res, stack_res])
print(f"\n{'='*55}")
print("📊 RESULTS WITH CV health_score (10 features)")
print('='*55)
print(results.to_string(index=False))
results.to_csv(f'{outputs}/model_results_cv.csv', index=False)

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['#2e8b57','#e07b39','#5b7fa6','#8b2e8b']
for i, metric in enumerate(['MAE','RMSE','R²']):
    axes[i].bar(results['Model'], results[metric],
                color=colors, edgecolor='white', width=0.5)
    axes[i].set_title(metric, fontsize=13, fontweight='bold')
    axes[i].set_ylim(0, results[metric].max() * 1.3)
    for j, v in enumerate(results[metric]):
        axes[i].text(j, v+0.005, str(v), ha='center',
                     fontsize=9, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20)
plt.suptitle('Model Comparison — With CV Health Score (10 Features)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{outputs}/model_comparison_cv.png', dpi=150)
plt.show()
print("\n Done! Paste results here.")

In [ ]:
# CELL 20H — Save updated Gradio app with CV image upload tab

PROJECT_DIR = '/content/drive/MyDrive/CropYieldProject'

app_code = """
import gradio as gr
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import os
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
from PIL import Image

BASE = os.path.dirname(os.path.abspath(__file__))

# Load ML models
rf    = joblib.load(os.path.join(BASE, "rf_cv.pkl"))
xgb   = joblib.load(os.path.join(BASE, "xgb_cv.pkl"))
svr   = joblib.load(os.path.join(BASE, "svr_cv.pkl"))
scaler= joblib.load(os.path.join(BASE, "scaler_cv.pkl"))
stack = joblib.load(os.path.join(BASE, "stack_cv.pkl"))

# Load CV model
cv_model = load_model(os.path.join(BASE, "leaf_classifier.h5"))

FEATURES = ['pesticides_tonnes', 'avg_temp', 'N', 'P', 'K',
            'temperature', 'humidity', 'ph', 'NPK_total', 'health_score']

def classify_leaf(image):
    if image is None:
        return 0.5, "No image — using default health score 0.5"
    img = Image.fromarray(image).resize((224, 224))
    arr = img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)
    score = float(cv_model.predict(arr, verbose=0)[0][0])
    label = "Healthy 🟢" if score > 0.5 else "Diseased 🔴"
    return round(score, 4), f"{label} (confidence: {score:.2%})"

def predict_yield(image, pesticides, avg_temp, N, P, K,
                  temperature, humidity, ph):

    health_score, health_label = classify_leaf(image)
    NPK_total = N + P + K

    input_data = np.array([[pesticides, avg_temp, N, P, K,
                            temperature, humidity, ph,
                            NPK_total, health_score]])
    input_df = pd.DataFrame(input_data, columns=FEATURES)
    input_sc = scaler.transform(input_df)

    rf_pred    = round(float(rf.predict(input_df)[0]), 3)
    xgb_pred   = round(float(xgb.predict(input_df)[0]), 3)
    svr_pred   = round(float(svr.predict(input_sc)[0]), 3)
    stack_pred = round(float(stack.predict(input_df)[0]), 3)

    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 3))
    models_list = ["Random Forest", "XGBoost", "SVR", "Ensemble"]
    preds_list  = [rf_pred, xgb_pred, svr_pred, stack_pred]
    colors      = ["#2e8b57", "#e07b39", "#5b7fa6", "#8b2e8b"]
    bars = ax.barh(models_list, preds_list, color=colors, edgecolor="white")
    ax.set_xlabel("Predicted Yield (Ton/Ha)")
    ax.set_title("Crop Yield Predictions — Punjab, Pakistan")
    for bar, val in zip(bars, preds_list):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                str(val), va="center", fontweight="bold")
    plt.tight_layout()

    # SHAP
    explainer = shap.TreeExplainer(xgb)
    shap_vals = explainer.shap_values(input_df)
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_vals[0],
            base_values=explainer.expected_value,
            data=input_df.iloc[0],
            feature_names=FEATURES),
        show=False)
    plt.tight_layout()

    result = (
        "🌿 Leaf Health    : " + health_label + chr(10) +
        "🌲 Random Forest  : " + str(rf_pred)    + " Ton/Ha" + chr(10) +
        "⚡ XGBoost        : " + str(xgb_pred)   + " Ton/Ha" + chr(10) +
        "🔵 SVR            : " + str(svr_pred)   + " Ton/Ha" + chr(10) +
        "🏆 Ensemble       : " + str(stack_pred) + " Ton/Ha"
    )
    return result, fig, fig2


with gr.Blocks(title="Crop Yield Predictor — Punjab Pakistan") as demo:

    gr.Markdown("# 🌾 Crop Yield Prediction — Punjab, Pakistan")
    gr.Markdown("### Multi-Model Ensemble + Computer Vision Leaf Health Analysis")
    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📷 Leaf Health (Computer Vision)")
            image_input = gr.Image(label="Upload Crop Leaf Image (optional)",
                                   type="numpy")
            gr.Markdown("*Upload a maize leaf image — CV model detects healthy/diseased*")

        with gr.Column():
            gr.Markdown("### 🌡️ Climate Parameters")
            pesticides  = gr.Slider(0, 30000, value=6000,
                                    label="Pesticides (tonnes)")
            avg_temp    = gr.Slider(15.0, 35.0, value=24.0,
                                    label="Average Temperature (C)")

        with gr.Column():
            gr.Markdown("### 🌱 Soil Parameters")
            N           = gr.Slider(0, 140, value=56,  label="Nitrogen (N)")
            P           = gr.Slider(0, 100, value=42,  label="Phosphorus (P)")
            K           = gr.Slider(0, 120, value=61,  label="Potassium (K)")
            temperature = gr.Slider(15.0, 35.0, value=24.0,
                                    label="Soil Temperature (C)")
            humidity    = gr.Slider(50.0, 100.0, value=86.0,
                                    label="Humidity (%)")
            ph          = gr.Slider(4.0, 9.0, value=6.4, label="Soil pH")

    predict_btn = gr.Button("🔍 Predict Yield", variant="primary")

    gr.Markdown("### 📊 Results")
    with gr.Row():
        result_text = gr.Textbox(label="Predictions", lines=6)
    with gr.Row():
        chart1 = gr.Plot(label="Model Comparison")
        chart2 = gr.Plot(label="SHAP Feature Importance")

    predict_btn.click(
        fn=predict_yield,
        inputs=[image_input, pesticides, avg_temp,
                N, P, K, temperature, humidity, ph],
        outputs=[result_text, chart1, chart2]
    )

    gr.Markdown("---")
    gr.Markdown(
        "**Region:** Bahawalpur, Punjab Pakistan | "
        "**CV Model:** MobileNetV2 (99.87% accuracy) | "
        "**Ensemble R²:** 0.9906"
    )

demo.launch()
"""

with open(f'{PROJECT_DIR}/app_cv.py', 'w') as f:
    f.write(app_code)

# Updated requirements
req = """gradio
numpy
pandas
scikit-learn
xgboost
shap
matplotlib
joblib
tensorflow
Pillow
"""
with open(f'{PROJECT_DIR}/requirements_cv.txt', 'w') as f:
    f.write(req)

print(" app_cv.py saved!")
print(" requirements_cv.txt saved!")

# Download both
from google.colab import files
files.download(f'{PROJECT_DIR}/app_cv.py')
files.download(f'{PROJECT_DIR}/requirements_cv.txt')

# Also download new models
files.download(f'{PROJECT_DIR}/models/rf_cv.pkl')
files.download(f'{PROJECT_DIR}/models/xgb_cv.pkl')
files.download(f'{PROJECT_DIR}/models/svr_cv.pkl')
files.download(f'{PROJECT_DIR}/models/scaler_cv.pkl')
files.download(f'{PROJECT_DIR}/models/stack_cv.pkl')
files.download(f'{PROJECT_DIR}/cv_module/model/leaf_classifier.h5')

print("\n Files to upload to Hugging Face:")
print("  app_cv.py          → rename to app.py")
print("  requirements_cv.txt → rename to requirements.txt")
print("  rf_cv.pkl")
print("  xgb_cv.pkl")
print("  svr_cv.pkl")
print("  scaler_cv.pkl")
print("  stack_cv.pkl")
print("  leaf_classifier.h5")